In [ ]:
# Assignment 3F: Sign Language Anomaly Detection

# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow import keras
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D

# 1. Load datasets
train_data = pd.read_csv("sign_mnist_train.csv")
test_data = pd.read_csv("sign_mnist_test.csv")

# 2. Separate features and labels
X_train = train_data.drop('label', axis=1).values
y_train = train_data['label'].values

X_test = test_data.drop('label', axis=1).values
y_test = test_data['label'].values

# 3. Reshape images and normalize pixel values
X_train = X_train.reshape(-1, 28, 28, 1) / 255.0
X_test = X_test.reshape(-1, 28, 28, 1) / 255.0

# 4. Select normal images
anomaly_class = 24

X_train = X_train[y_train != anomaly_class]
y_test = (y_test == anomaly_class).astype(int)

print("Normal Training Images:", X_train.shape)
print("Testing Images:", X_test.shape)

# 5. Build Convolutional Autoencoder
autoencoder = Sequential([
    Input(shape=(28, 28, 1)),
    

    # Encoder
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2), padding='same'),
    Conv2D(16, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2), padding='same'),

    # Decoder
    Conv2D(16, (3, 3), activation='relu', padding='same'),
    UpSampling2D((2, 2)),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    UpSampling2D((2, 2)),
    
    Conv2D(1, (3, 3), activation='sigmoid', padding='same')
])

# 6. Compile model
autoencoder.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# 7. Train model
history = autoencoder.fit(
    X_train, X_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2
)

print("Training Completed!")

# 8. Calculate Training Reconstruction Error
train_pred = autoencoder.predict(X_train)

train_error = np.mean(
    np.square(X_train - train_pred), axis=(1, 2, 3)
)

# 9. Set Threshold
threshold = np.percentile(train_error, 98)

print("Threshold:", threshold)

# 10. Calculate Testing Reconstruction Error
test_pred = autoencoder.predict(X_test)

test_error = np.mean(
    np.square(X_test - test_pred), axis=(1, 2, 3)
)

# 11. Detect Anomaly
y_pred = (test_error > threshold).astype(int)

print("Predicted Anomalies:", np.sum(y_pred == 1))
print("Predicted Normal:", np.sum(y_pred == 0))

# 12. Evaluate Model
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 13. Plot Training Loss
sns.lineplot(data=pd.DataFrame(history.history)[['loss', 'val_loss']])
plt.title("Autoencoder Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("MSE Loss")
plt.show()